In [1]:
from helpers_ps.bloomberg.bbg import bdh_many, _to_pandas_df

In [2]:
from xbbg import blp

In [35]:
b = blp.bdh(tickers="SPX INDEX", flds="BEST_PE_RATIO", start_date="2025-12-31", end_date="2026-01-31", BEST_FPERIOD_OVERRIDE="BF", Per="M")
b = _to_pandas_df(b)

In [36]:
b

,ticker,date,field,value
0,SPX INDEX,2025-12-31,BEST_PE_RATIO,22.0252
1,SPX INDEX,2026-01-30,BEST_PE_RATIO,21.8742


In [14]:

df = blp.bdh(
    tickers="SPX INDEX",
    flds="BEST_PE_RATIO",
    start_date="2025-04-30",
    end_date="2026-05-31"
)

print(type(df))
print(df)


<class 'narwhals.stable.v1.DataFrame'>
┌───────────────────────────────────────┐
|          Narwhals DataFrame           |
| Use `.to_native` to see native output |
└───────────────────────────────────────┘


In [39]:
a, err = bdh_many([dict(
    ticker="SPX INDEX",
    flds="BEST_PE_RATIO",
    start_date="2025-12-31",
    end_date="2026-05-31",
    BEST_FPERIOD_OVERRIDE="BF",
    Days="C",
    Fill="P"
),
dict(
    ticker="NDX INDEX",
    flds="BEST_PE_RATIO",
    start_date="2025-12-31",
    end_date="2026-05-31",
    BEST_FPERIOD_OVERRIDE="BF",
    Per="M",
    Days="C",
    Fill="P"
)
], errors=True)

In [40]:
import pandas as pd

In [41]:
#a["date"] = pd.to_datetime(a["date"])
c = a[a["date"] >= "2025-12-31"]
c

,date,ticker,field,value,overrides
0,2025-12-31,NDX INDEX,BEST_PE_RATIO,24.9316,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'..."
1,2026-01-30,NDX INDEX,BEST_PE_RATIO,24.6793,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'..."
2,2026-02-27,NDX INDEX,BEST_PE_RATIO,23.8211,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'..."
3,2026-03-31,NDX INDEX,BEST_PE_RATIO,20.9737,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'..."
4,2026-04-30,NDX INDEX,BEST_PE_RATIO,22.9464,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'..."
...,...,...,...,...,...
153,2026-05-27,SPX INDEX,BEST_PE_RATIO,20.9498,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'"
154,2026-05-28,SPX INDEX,BEST_PE_RATIO,21.0543,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'"
155,2026-05-29,SPX INDEX,BEST_PE_RATIO,21.0762,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'"
156,2026-05-30,SPX INDEX,BEST_PE_RATIO,21.0665,"BEST_FPERIOD_OVERRIDE='BF', Days='C', Fill='P'"


In [5]:
raw = blp.bdh(
    tickers="SPY US Equity",
    flds="PX_LAST",
    start_date="2025-04-30",
    end_date="2026-05-31"
)

df = _to_pandas_df(raw)

print("TYPE:", type(df))
print("INDEX:", df.index)
print("INDEX NAME:", df.index.name)
print("COLUMNS:", df.columns)
print("COLUMNS LIST:", list(df.columns))
print(df.head())

TYPE: <class 'pandas.DataFrame'>
INDEX: RangeIndex(start=0, stop=272, step=1)
INDEX NAME: None
COLUMNS: Index(['ticker', 'date', 'field', 'value'], dtype='str')
COLUMNS LIST: ['ticker', 'date', 'field', 'value']
          ticker        date    field     value
0  SPY US Equity  2025-04-30  PX_LAST  546.8455
1  SPY US Equity  2025-05-01  PX_LAST  550.7210
2  SPY US Equity  2025-05-02  PX_LAST  558.8960
3  SPY US Equity  2025-05-05  PX_LAST  555.6911
4  SPY US Equity  2025-05-06  PX_LAST  551.0464


In [42]:
list(set(["unique_id","grouping", "ticker", "field", "start_date", "overrides"]))

['grouping', 'overrides', 'start_date', 'field', 'ticker', 'unique_id']

In [43]:
a = {
    "day": "5",
    "prueba": "2"
}

In [45]:
b = {
    "2":"ijijoij",
    **a
}

In [46]:
b

{'2': 'ijijoij', 'day': '5', 'prueba': '2'}

In [ ]:
def _to_dict(data:str):
    data = data.split(",")
    final_dict = {}
    for d in data:
        _key = d.split("=")[0]
        _val = d.split("=")[1]
        final_dict[_key] = _val
    return final_dict

def bdh_many_data_frame(dataframe: pd.DataFrame, cache: bool = True, save_dict: dict = None) -> pd.DataFrame:
    """
    Funcion para pasar la plantilla de dataframe de base de tickers las unicas columnas permitidas son
    "unique_id","grouping", "ticker", "field", "start_date" y cualquier otra columna de override
    """

    # Columas necesarias que esten en el dataframe
    columns_necesary = list(set(["unique_id","grouping", "ticker", "field", "start_date", "overrides"]))
    
    # Validación de las columnas solo tengan las columnas requeridas
    columns_df = list(set([x for x in dataframe.columns.tolist() if x in columns_necesary]))

    if columns_necesary != columns_df:
        raise TypeError(f"Las unicas columnas que deben estar en el data frame son {columns_necesary}\nSin embargo las columnas recibidas son {columns_df}")

    #filtrando solo por las columnas necesarias
    dataframe = dataframe[columns_necesary]
    
    # primero haremos una pre validación de que la data que estamos buscando traer no existe previamente en el 
    if cache:
        pass
    else:
        # construimos los dict basado en los grouping de overrides
        grouping = dataframe["grouping"].unique()

        list_dicts = []

        for group in grouping:
            _df = dataframe[dataframe["grouping"] == group].copy()
            overrides = _to_dict(_df["overrides"].unique()[0])
            final_dict = dict(
                tickers = _df["ticker"].tolist(),
                field = _df["field"].unique()[0],
                start_date = _df["start_date"].unique()[0],
                **overrides
            )
            list_dicts.append(final_dict)
        
        return list_dicts